# API exploration

Use the devcontainer, run `make demo`, and select the project `.venv` kernel. For an existing container, run `make notes-install` once. Run cells in order. This notebook reuses `TasksAPI` through a worker session; the API workflow does not launch a browser. Close the session before rerunning setup.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from notebooks import NotebookSession

import os
from environment import load_environment

load_environment()
BASE_URL = os.environ.get("BASE_URL", "http://127.0.0.1:8765")  # Start `make demo` in another terminal.


In [ ]:
session = await NotebookSession.api(BASE_URL)


## Inspect live data

Convert responses to plain data inside `session.run`. Add your own assertions in the cell.


In [ ]:
tasks = await session.run(lambda api: api.list().json())
assert isinstance(tasks, list)
display(tasks)


In [ ]:
def create_and_inspect(api):
    response = api.create("Explore API data")
    assert response.status == 201
    return {"status": response.status, "headers": response.headers, "data": response.json()}

created = await session.run(create_and_inspect)
assert created["data"]["title"] == "Explore API data"
display(created)


In [ ]:
task_id = created["data"]["id"]
fetched = await session.run(lambda api: api.get(task_id).json())
assert fetched == created["data"]
display(fetched)


## Export and clean up

Only this notebook's created task is deleted. Exports go to ignored `notes/output/`.


In [ ]:
import json
output = ROOT / "notes" / "output"
output.mkdir(exist_ok=True)
(output / "api-task.json").write_text(json.dumps(fetched, indent=2))


In [ ]:
status = await session.run(lambda api: api.delete(task_id).status)
assert status == 204
await session.close()
